# MindFlow Step 6a · Market regime (VIX) for volatility

**Question.** The vol label uses each stock's fixed training-period median |r|, so the volatility regime is part of what it measures. Do the VIX and multi-day realized volatility raise HGB's AUC beyond the 38 hand + SPY numbers?


HGB only (the strongest model so far, Step 3a: 0.689 / 0.679), same rows for every feature set, scored on 2023 with a
paired day bootstrap against the 38-feature model. The test years are not used.

| feature set | what is added to the 38 hand + SPY numbers |
|---|---|
| `+ VIX` | previous session's VIX close (log), its 1- and 5-session change, distance to its 20-session mean |
| `+ multi-day RV` | stock realized vol over the last 5 and 20 completed sessions, their ratio, and SPY's 5/20-session RV |
| `+ VIX + multi-day RV` | both |
| `regime only` | the 9 regime features alone (no 38) |
| `+ VIXY intraday` | VIXY over the same 78-bar window as SPY: returns, cumulative returns, realized vol (needs `vixy_panel_v4.zip`) |

VIX comes from Cboe's public daily file (downloaded once, cached in `MindFlow/data/`). Only closes of sessions **before**
the decision day are used. Realized vol uses intraday 5-minute returns of completed sessions only (no overnight gap,
so no ex-date is ever inside a return). CPU is enough; turn on High-RAM (the bundle's price arrays need ~6 GB).


<details><summary>Where this notebook sits in the MindFlow</summary>


| step | question | notebooks |
|---|---|---|
| **1** | Does the JKX chart signal exist at the daily frequency on our large caps (also for pretrained models)? | `Step1a_Daily_JKX_CNN`, `Step1b_Daily_Pretrained` |
| **2** | Moved to 5-minute charts: does any model (CNN, ResNet, 1D CNN, ViT, HGB, frozen DINOv2/CLIP, time-series foundation models) find direction? | `Step2a_Intraday_Direction_Models`, `Step2b_Intraday_Direction_Pretrained` |
| **3** | Same models, target = volatility: is there a signal, and who wins? | `Step3a_Intraday_Volatility_Models`, `Step3b_Intraday_Volatility_Pretrained` |
| **4** | Add information to the picture step by step until the image model reaches HGB; is the image still useful beyond the features; which features matter? | `Step4a_Add_Image_Information`, `Step4b_Feature_Selection` |
| **5** | One figure and one parameter table over all steps | `Step5_Summary` |
| **6** | Next questions: does market-regime information (VIX, multi-day realized volatility) help? | `Step6a_VIX_Regime`, `Step6b_Final_Model` (final model + held-out test), `Step6c_Volatility_Use` (IC, long-short, risk sizing), `Step6d_Pure_Image` (the numbers drawn into the picture, no numbers input) |
| **7** | Efficient computing for the final model: 16-bit / INT8 inference, LoRA fine-tuning of a pretrained ViT | `Step7a_Efficient_Computing`, `Step7b_Distillation` (small students taught by the final CNN) |

</details>

**Data.** `alldata_bundle_v4.zip` (TAQ filter v4, point-in-time price adjustment): S&P 500 ∪ NASDAQ 100 ∪ DJIA 30
members by date, 2,514 sessions 2016-01-04 … 2025-12-31, five-minute bars plus SPY on the same clock.
**Split.** Train 2016–2022 · validation 2023 (every choice) · test 2024–2025 (last section, run once at the end).
**Advisor rule.** Every model prints its parameter count; for HGB the count is tree nodes.

## 1. Setup
Put `alldata_bundle_v4.zip` and the `code/` folder in `Demo/MindFlow` on Drive. The bundle (1.3 GB) is copied to the
Colab disk and unzipped once per runtime; the next notebook in the same runtime reuses it.

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

LOCAL_TEST = os.environ.get("MINDFLOW_LOCAL_TEST") == "1"          # repository QA only (synthetic data)
if LOCAL_TEST:
    HERE, LOCAL_ROOT = Path(os.environ["MINDFLOW_HERE"]), Path(os.environ["MINDFLOW_LOCAL"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE, LOCAL_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo/MindFlow"), Path("/content/mindflow")
CODE = HERE / "code"
assert (CODE / "mindflow.py").is_file(), f"Put the code/ folder in {HERE}"
sys.path[:0] = [str(CODE)]
import mindflow as mf
PREPARED = mf.stage_bundle(HERE / mf.BUNDLE_NAME, LOCAL_ROOT)
PANELS_ADDED = mf.stage_panels(HERE, PREPARED)          # e.g. vixy_panel_v4.zip, if uploaded
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
import chartcnn
import feature_select as fs
import regime as rg
print("regime", rg.__version__, "| feature_select", fs.__version__)
DEVICE = torch.device("cpu")

## 2. Configuration
**Time (measured, Colab CPU High-RAM):** ~45 min for the 5 feature sets; with the VIXY panel (7 sets) ~60 min. `QUICK = True`: 300,000 rows, ~15 min.

In [ ]:
SMOKE = LOCAL_TEST
RUN_ID = None
QUICK = False
HORIZONS = [1, 12]                                 # bars of 5 minutes
SETTINGS = dict(stride=12, max_train=1_000_000, min_bars=60, seed=69)
if QUICK:
    SETTINGS.update(max_train=300_000)
if SMOKE:
    SETTINGS.update(max_train=3_000, min_bars=10)
CELLS = [dict(name=f"vol_h{h}", kind="vol", target="vol", horizon=h, minutes=5 * h, stride=SETTINGS["stride"]) for h in HORIZONS]
RUN_ID, RUN_DIR, RESUME = mf.new_run(HERE, "6a", RUN_ID, "regime", LOCAL_TEST)
mf.save_settings(RUN_DIR, RESUME, settings=SETTINGS, cells=CELLS, quick=QUICK, regime=rg.__version__)
print("RUN_ID:", RUN_ID, "->", RUN_DIR)

## 3. Regime features (VIX and multi-day realized volatility)

In [ ]:
store = chartcnn.Store(PREPARED)
VIX_CACHE = HERE / "data" / "VIX_History.csv"
vix = rg.load_vix(VIX_CACHE)
print(f"VIX: {len(vix):,} closes, {vix.index[0]} … {vix.index[-1]} (cached at {VIX_CACHE})")
sessions = np.unique(store.dates)
missing = [d for d in sessions if d not in vix.index]
print(f"bundle sessions without a VIX close: {len(missing)}", missing[:5])
VIXF = rg.vix_table(vix, sessions)
VIXY = rg.load_vixy(PREPARED)                     # VIXY intraday features if vixy_panel_v4.zip is in Demo/MindFlow
print("VIXY intraday features:", "yes" if VIXY is not None else "no (vixy_panel_v4.zip not found)")
t0 = time.perf_counter()
RV = rg.rv_tables(store, SETTINGS["min_bars"])
print(f"realized-vol tables in {(time.perf_counter() - t0) / 60:.1f} min:", {k: v.shape for k, v in RV.items()})
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
days = pd.to_datetime(sessions)
ax[0].plot(days, np.exp(VIXF.vix_log.to_numpy()), lw=0.8); ax[0].set_ylabel("VIX (prev. close)")
if "spy_rv20d" in RV:
    ax[1].plot(days, RV["spy_rv20d"][:len(days)], lw=0.8, c="tab:orange"); ax[1].set_ylabel("SPY RV, 20 sessions")
for a in ax:
    a.axvspan(pd.Timestamp(store.config["train_end"]), pd.Timestamp(store.config["validation_end"]), color="gray", alpha=0.15)
ax[0].set_title("regime features (grey = validation 2023)")
fig.tight_layout(); fig.savefig(RUN_DIR / "regime_series.png", dpi=120); plt.show()

## 4. HGB with and without regime features, per cell

In [ ]:
ALL = {}
for cell in CELLS:
    out = RUN_DIR / cell["name"]; out.mkdir(exist_ok=True)
    if (out / "comparison.csv").exists():
        print(cell["name"], "already done"); ALL[cell["name"]] = {k: pd.read_csv(out / f"{k}.csv") for k in ("comparison", "by_regime", "importance")}
        continue
    print("\n" + "=" * 20, cell["name"], "=" * 20)
    t0 = time.perf_counter()
    d, X = rg.build(store, cell, SETTINGS, RV, VIXF, VIXY)
    base = [c for c in X["train"].columns if c not in sum(rg.GROUPS.values(), []) + rg.VIXY_COLS]
    print(f"train {len(d['train']):,} | validation {len(d['valid']):,} on {d['valid'].decision_date.nunique()} days | "
          f"{len(base)} base + {X['train'].shape[1] - len(base)} regime features")
    print("missing share of regime features (validation):", X["valid"][sum(rg.GROUPS.values(), [])].isna().mean().round(3).to_dict())
    subsets = {"38 hand + SPY": base,
               "+ VIX": base + rg.VIX_COLS,
               "+ multi-day RV": base + rg.GROUPS["multi-day RV"],
               "+ VIX + multi-day RV": base + sum(rg.GROUPS.values(), []),
               "regime only": sum(rg.GROUPS.values(), [])}
    if VIXY is not None:
        subsets["+ VIXY intraday"] = base + rg.VIXY_COLS
        subsets["+ VIXY intraday + multi-day RV"] = base + rg.VIXY_COLS + rg.GROUPS["multi-day RV"]
    comp, scores, models = rg.compare(d, X, subsets, SETTINGS["seed"], n_boot=50 if SMOKE else 300)
    display(comp.round(4))
    reg = rg.by_regime(d["valid"], scores, X["valid"].vix_log.to_numpy(), list(subsets))
    print("AUC within VIX terciles of 2023 (previous close):"); display(reg.round(4))
    full = "+ VIXY intraday + multi-day RV" if VIXY is not None else "+ VIX + multi-day RV"
    _, imp = fs.permutation_importance(models[full], X["valid"][subsets[full]], d["valid"].label.to_numpy(),
                                       1 if SMOKE else 3, SETTINGS["seed"])
    res = dict(comparison=comp, by_regime=reg, importance=imp)
    for k, v in res.items():
        v.to_csv(out / f"{k}.csv", index=False)
    ALL[cell["name"]] = res
    print(f"{cell['name']} done in {(time.perf_counter() - t0) / 60:.1f} min")

## 5. Summary

In [ ]:
rows = []
for name, r in ALL.items():
    t = r["comparison"].assign(cell=name)
    rows.append(t)
SUMMARY = pd.concat(rows)[["cell", "model", "n_features", "tree_nodes", "auc", "auc_low", "auc_high", "diff_vs_ref",
                           "diff_low", "diff_high", "verdict"]]
SUMMARY.to_csv(RUN_DIR / "summary.csv", index=False)
display(SUMMARY.round(4))
for name, r in ALL.items():
    imp = r["importance"]
    reg = imp[imp.feature.isin(sum(rg.GROUPS.values(), []) + rg.VIXY_COLS)]
    print(f"{name}: regime features by permutation importance (2023, full model):")
    display(reg.round(5))
lines = []
for name, r in ALL.items():
    t = r["comparison"].set_index("model")
    for m in ("+ VIX", "+ multi-day RV", "+ VIX + multi-day RV", "+ VIXY intraday", "+ VIXY intraday + multi-day RV"):
        if m not in t.index:
            continue
        x = t.loc[m]
        lines.append(f"- {name} {m}: AUC {x.auc:.4f} vs {t.loc['38 hand + SPY'].auc:.4f}, "
                     f"diff {x.diff_vs_ref:+.4f} [{x.diff_low:+.4f}, {x.diff_high:+.4f}] -> {x.verdict}")
TAKEAWAYS = "\n".join(lines)
(RUN_DIR / "takeaways.md").write_text(TAKEAWAYS + "\n"); print(TAKEAWAYS)

### How to read it
- **`+ VIX` better**: the market's expected volatility adds what one session of bars cannot see; worth adding to the
  CNN's feature branch (Step 4) and trying the VIXY intraday panel.
- **`+ multi-day RV` better but `+ VIX` tied**: the stock's own slower volatility is what matters; VIX adds nothing.
- **All tied**: the 78-bar window already captures the regime; VIX is not worth the extra data.
- **By VIX tercile**: the share of positive labels should rise with VIX (fixed threshold); a model whose AUC drops in
  the high tercile is mis-calibrated across regimes.